In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

LOGS = Path("../data/captures")

scan = pd.read_json(
    LOGS / "zeek_portscan_01" / "conn.log",
    lines=True
)

benign = pd.read_json(
    LOGS / "zeek_benign_01" / "conn.log",
    lines=True
)

CIC_FILE = Path(
    "../data/MachineLearningCSV/MachineLearningCVE/"
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv"
)

cic = pd.read_csv(CIC_FILE)
cic.columns = cic.columns.str.strip()

print("Zeek scan:", scan.shape)
print("Zeek benign:", benign.shape)
print("CIC dataset:", cic.shape)
print(cic["Label"].value_counts())

Zeek scan: (65554, 21)
Zeek benign: (3, 21)
CIC dataset: (286467, 79)
Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64


In [3]:
def zeek_to_common(conn: pd.DataFrame) -> pd.DataFrame:
    d = conn.copy()

    numeric_cols = [
        "duration", "orig_bytes", "resp_bytes",
        "orig_pkts", "resp_pkts", "id.resp_p"
    ]

    for col in numeric_cols:
        d[col] = pd.to_numeric(d[col], errors="coerce")

    d[numeric_cols] = d[numeric_cols].fillna(0)

    hist = d["history"].fillna("").astype(str)
    total_s = d["duration"].replace(0, np.nan)

    out = pd.DataFrame({
        "dst_port": d["id.resp_p"],
        "duration_us": d["duration"] * 1_000_000,
        "fwd_pkts": d["orig_pkts"],
        "bwd_pkts": d["resp_pkts"],
        "fwd_bytes": d["orig_bytes"],
        "bwd_bytes": d["resp_bytes"],
        "bytes_per_s": (
            (d["orig_bytes"] + d["resp_bytes"]) / total_s
        ).fillna(0),
        "pkts_per_s": (
            (d["orig_pkts"] + d["resp_pkts"]) / total_s
        ).fillna(0),
        "syn_count": hist.str.count("[SsHh]"),
        "rst_count": hist.str.count("[Rr]"),
        "fin_count": hist.str.count("[Ff]"),
    })

    return out

In [4]:
CIC_RENAME = {
    "Destination Port": "dst_port",
    "Flow Duration": "duration_us",
    "Total Fwd Packets": "fwd_pkts",
    "Total Backward Packets": "bwd_pkts",
    "Total Length of Fwd Packets": "fwd_bytes",
    "Total Length of Bwd Packets": "bwd_bytes",
    "Flow Bytes/s": "bytes_per_s",
    "Flow Packets/s": "pkts_per_s",
    "SYN Flag Count": "syn_count",
    "RST Flag Count": "rst_count",
    "FIN Flag Count": "fin_count",
}

def cic_to_common(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns=lambda c: c.strip())

    out = df[list(CIC_RENAME)].rename(columns=CIC_RENAME)

    for col in out.columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    return out.replace([np.inf, -np.inf], np.nan)

In [5]:
zeek_scan = zeek_to_common(scan)
zeek_benign = zeek_to_common(benign)

cic_common = cic_to_common(cic)

cic_scan = cic_common.loc[
    cic["Label"].str.strip().eq("PortScan")
].copy()

cic_benign = cic_common.loc[
    cic["Label"].str.strip().eq("BENIGN")
].copy()

print("Zeek scan:", zeek_scan.shape)
print("CIC scan:", cic_scan.shape)
print("Zeek benign:", zeek_benign.shape)
print("CIC benign:", cic_benign.shape)

Zeek scan: (65554, 11)
CIC scan: (158930, 11)
Zeek benign: (3, 11)
CIC benign: (127537, 11)


In [6]:
features = ["fwd_pkts", "duration_us", "syn_count"]

median_comparison = pd.DataFrame({
    "Zeek port scan": zeek_scan[features].median(),
    "CIC port scan": cic_scan[features].median(),
    "Zeek benign": zeek_benign[features].median(),
    "CIC benign": cic_benign[features].median(),
})

median_comparison

,Zeek port scan,CIC port scan,Zeek benign,CIC benign
fwd_pkts,1.000000,1.0,1.700000e+01,2.0
duration_us,28555.512428,47.0,1.718235e+07,30964.0
syn_count,1.000000,0.0,2.000000e+00,0.0


In [8]:
print("Missing values in Zeek scan:")
print(zeek_scan[features].isna().sum())

print("\nMissing values in CIC scan:")
print(cic_scan[features].isna().sum())

print("\nInfinite values in CIC scan:")
print(np.isinf(cic_scan[features]).sum())

Missing values in Zeek scan:
fwd_pkts       0
duration_us    0
syn_count      0
dtype: int64

Missing values in CIC scan:
fwd_pkts       0
duration_us    0
syn_count      0
dtype: int64

Infinite values in CIC scan:
fwd_pkts       0
duration_us    0
syn_count      0
dtype: int64


In [9]:
print("ZEek scan summary:")
print(zeek_scan[["fwd_pkts", "duration_us", "syn_count"]].describe())

print("\nCIC scan summary:")
print(cic_scan[["fwd_pkts", "duration_us", "syn_count"]].describe())

ZEek scan summary:
           fwd_pkts   duration_us     syn_count
count  65554.000000  6.555400e+04  65554.000000
mean       1.001480  3.576154e+04      1.000259
std        0.058044  4.076296e+05      0.023105
min        1.000000  0.000000e+00      0.000000
25%        1.000000  3.749132e+03      1.000000
50%        1.000000  2.855551e+04      1.000000
75%        1.000000  5.353349e+04      1.000000
max        7.000000  8.068117e+07      2.000000

CIC scan summary:
            fwd_pkts   duration_us  syn_count
count  158930.000000  1.589300e+05   158930.0
mean        1.017580  8.282023e+04        0.0
std         0.425594  2.325855e+06        0.0
min         1.000000  0.000000e+00        0.0
25%         1.000000  4.200000e+01        0.0
50%         1.000000  4.700000e+01        0.0
75%         1.000000  6.100000e+01        0.0
max       150.000000  1.198097e+08        0.0


In [10]:
print("Zeek history examples:")
print(scan[["history", "conn_state", "duration"]].head(10))

print("\nCIC SYN Flag Count distribution:")
print(cic.loc[
    cic["Label"].str.strip().eq("PortScan"),
    "SYN Flag Count"
].value_counts(dropna=False).head(10))

Zeek history examples:
  history conn_state  duration
0     NaN        OTH       NaN
1     NaN        OTH  4.225890
2       D         S0       NaN
3      Sr        REJ  0.000647
4      Sr        REJ  0.002314
5      Sr        REJ  0.002314
6      Sr        REJ  0.002314
7      Sr        REJ  0.002314
8    ShAR       RSTO  0.002315
9      Sr        REJ  0.004663

CIC SYN Flag Count distribution:
SYN Flag Count
0    158930
Name: count, dtype: int64


### D1 — Initial Feature Comparison

The initial comparison found that the median forward packet count matched for the port-scan data, but differed for benign traffic. The duration medians differed substantially between Zeek and CIC-IDS2017, and the SYN-count medians also differed.

The tested features contained no missing or infinite values in the displayed comparison. However, the differences show that the mapping needs further validation before model training. The benign Zeek capture is also very small, so its results should be treated as preliminary.

Next, the original Zeek history values and CIC SYN flag counts will be inspected to investigate the differences.

In [11]:
features_to_check = [
    "dst_port",
    "fwd_pkts",
    "bwd_pkts",
    "fwd_bytes",
    "bwd_bytes",
    "bytes_per_s",
    "pkts_per_s",
    "rst_count",
    "fin_count",
]

comparison = pd.DataFrame({
    "Zeek scan": zeek_scan[features_to_check].median(),
    "CIC scan": cic_scan[features_to_check].median(),
    "Zeek benign": zeek_benign[features_to_check].median(),
    "CIC benign": cic_benign[features_to_check].median(),
})

comparison

,Zeek scan,CIC scan,Zeek benign,CIC benign
dst_port,32761.500000,3690.00000,22.000000,80.000000
fwd_pkts,1.000000,1.00000,17.000000,2.000000
bwd_pkts,1.000000,1.00000,18.000000,2.000000
fwd_bytes,0.000000,0.00000,76.000000,68.000000
bwd_bytes,0.000000,6.00000,197.000000,141.000000
bytes_per_s,0.000000,139534.88370,136.368134,5681.176299
pkts_per_s,63.754122,42553.19149,2.036974,112.633796
rst_count,1.000000,0.00000,0.000000,0.000000
fin_count,0.000000,0.00000,2.000000,0.000000
